# Qwen3-4B API trên Kaggle

Bật **GPU** và **Internet**, sau đó chạy các cell theo thứ tự. Thêm Kaggle Secrets `QWEN_API_TOKEN` (token ngẫu nhiên dài ít nhất 32 ký tự, không in ra màn hình) và `NGROK_AUTHTOKEN`, bật quyền truy cập cho notebook. `/health` và `/generate` đều yêu cầu bearer token. URL tunnel thay đổi khi khởi động lại; cập nhật `QWEN_API_URL` trong `.env`. Endpoint dành cho demo/evaluation, phụ thuộc phiên Kaggle và quyền truy cập ngrok.

Nguồn: [Qwen model card](https://huggingface.co/Qwen/Qwen3-4B), [OpenAI Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs).

In [ ]:
%pip install -q "transformers>=4.51,<5" "accelerate>=1,<2" "fastapi>=0.115,<1" "uvicorn>=0.30,<1" "pyngrok>=7,<8"


In [ ]:
from kaggle_secrets import UserSecretsClient

secrets_client = UserSecretsClient()
API_TOKEN = secrets_client.get_secret("QWEN_API_TOKEN")
NGROK_TOKEN = secrets_client.get_secret("NGROK_AUTHTOKEN")
assert len(API_TOKEN) >= 32, "Use a random bearer token with at least 32 characters."
MODEL_ID = "Qwen/Qwen3-4B"


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), "Enable a GPU accelerator in Kaggle settings."
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)
model.eval()

def generate_text(prompt, max_new_tokens, temperature):
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    input_length = inputs["input_ids"].shape[-1]
    if input_length + max_new_tokens > 32768:
        raise ValueError("Prompt exceeds model context budget")
    sampling = {"do_sample": temperature > 0}
    if temperature > 0:
        sampling.update(temperature=temperature, top_p=0.8, top_k=20)
    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens, **sampling,
            pad_token_id=tokenizer.eos_token_id,
        )
    answer = tokenizer.decode(output[0][input_length:], skip_special_tokens=True).strip()
    if not answer:
        raise RuntimeError("Empty generation")
    return answer


In [ ]:
import hmac
import threading
from fastapi import Depends, FastAPI, HTTPException
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from pydantic import BaseModel, Field

MODEL_NAME = "Qwen/Qwen3-4B"

class GenerateRequest(BaseModel):
    prompt: str = Field(min_length=1, max_length=60000)
    max_new_tokens: int = Field(default=1024, ge=1, le=4096)
    temperature: float = Field(default=0.1, ge=0, le=2)

class GenerateResponse(BaseModel):
    text: str
    model: str

def create_app(generator, api_token):
    app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)
    security = HTTPBearer(auto_error=False)
    generation_lock = threading.Lock()

    def authenticate(credentials: HTTPAuthorizationCredentials | None = Depends(security)):
        if credentials is None or not hmac.compare_digest(
            credentials.credentials.encode("utf-8"), api_token.encode("utf-8")
        ):
            raise HTTPException(status_code=401, detail="Unauthorized")

    @app.get("/health", dependencies=[Depends(authenticate)])
    def health():
        return {"status": "ok", "model": MODEL_NAME}

    @app.post("/generate", response_model=GenerateResponse, dependencies=[Depends(authenticate)])
    def generate(request: GenerateRequest):
        if not generation_lock.acquire(blocking=False):
            raise HTTPException(status_code=429, detail="Generation in progress")
        try:
            text = generator(request.prompt, request.max_new_tokens, request.temperature)
            return GenerateResponse(text=text, model=MODEL_NAME)
        except ValueError:
            raise HTTPException(status_code=413, detail="Prompt exceeds model context budget") from None
        except Exception:
            raise HTTPException(status_code=503, detail="Generation unavailable") from None
        finally:
            generation_lock.release()

    return app


In [ ]:
import time
import httpx
import uvicorn
from pyngrok import ngrok

# Stop the existing server/tunnel before rerunning this cell.
if "server" in globals() and server.started:
    server.should_exit = True
    server_thread.join(timeout=10)
    ngrok.kill()

app = create_app(generate_text, API_TOKEN)
server = uvicorn.Server(uvicorn.Config(
    app, host="127.0.0.1", port=8000, log_level="warning", access_log=False
))
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
for _ in range(120):
    if server.started:
        break
    if not server_thread.is_alive():
        raise RuntimeError("Server did not start")
    time.sleep(0.25)
else:
    raise RuntimeError("Server startup timed out")

# Check authentication and health locally before exposing the tunnel.
with httpx.Client(trust_env=False) as client:
    assert client.get("http://127.0.0.1:8000/health").status_code == 401
    health = client.get(
        "http://127.0.0.1:8000/health",
        headers={"Authorization": f"Bearer {API_TOKEN}"},
    )
    assert health.status_code == 200
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(addr="http://127.0.0.1:8000", proto="http", bind_tls=True)
assert tunnel.public_url.startswith("https://")
print("QWEN_API_URL=" + tunnel.public_url + "/generate")
print("Copy QWEN_API_TOKEN from your Kaggle Secret to local .env. Do not print it.")


Sau khi cập nhật `.env`, chạy `rag-answer examples/request.json` ở máy local để kiểm tra xuyên suốt. Khi kết thúc, chạy cell dưới đây để đóng tunnel và server. Nếu Kaggle dừng phiên, hãy chạy lại notebook và cập nhật URL.

In [ ]:
server.should_exit = True
server_thread.join(timeout=10)
ngrok.kill()
